In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_csv("../data/feat/go_emotions_with_features.csv")

golden_texts = pd.read_json("../data/golden_texts/golden_texts.json")

print("------------")
print("GoEmotions")
print("Shape: ", df.shape)
print("COLS")
print(df.columns)
display(df)

print("------------")
print("Golden Texts")
print("Shape: ", golden_texts.shape)
print("COLS")
print(golden_texts.columns)
display(golden_texts)

In [ ]:
TEXT_COL = "BASE_TEXT_PT"

emotions = golden_texts.columns.tolist()

def tokenize(text):
    return set(str(text).lower().split())

all_texts = df[TEXT_COL].tolist()

golden_flat = []
for emo in emotions:
    golden_flat.extend(golden_texts[emo].dropna().tolist())

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(all_texts + golden_flat)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

golden_embeddings = {}
for emo in emotions:
    texts = golden_texts[emo].dropna().tolist()
    emb = model.encode(texts, normalize_embeddings=True)
    golden_embeddings[emo] = emb

def tfidf_similarities(text):
    vec = tfidf_vectorizer.transform([text])
    results = {}

    for emo in emotions:
        g_texts = golden_texts[emo].dropna().tolist()
        g_vecs = tfidf_vectorizer.transform(g_texts)

        sims = cosine_similarity(vec, g_vecs)[0]
        results[f"{emo}_tfidf_mean"] = sims.mean()
        results[f"{emo}_tfidf_max"] = sims.max()

    return results

def embedding_similarities(text):
    emb = model.encode([text], normalize_embeddings=True)
    results = {}

    for emo in emotions:
        g_emb = golden_embeddings[emo]
        sims = cosine_similarity(emb, g_emb)[0]

        results[f"{emo}_emb_mean"] = sims.mean()
        results[f"{emo}_emb_max"] = sims.max()

    return results

def jaccard_similarities(text):
    tokens = tokenize(text)
    results = {}

    for emo in emotions:
        sims = []
        for g in golden_texts[emo].dropna():
            g_tokens = tokenize(g)
            inter = len(tokens & g_tokens)
            union = len(tokens | g_tokens)
            sims.append(inter / union if union > 0 else 0)

        sims = np.array(sims)
        results[f"{emo}_jaccard_mean"] = sims.mean()
        results[f"{emo}_jaccard_max"] = sims.max()

    return results

def bertscore_similarities(text):
    results = {}

    for emo in emotions:
        g_texts = golden_texts[emo].dropna().tolist()
        preds = [text] * len(g_texts)

        _, _, f1 = bertscore(preds, g_texts, lang="pt", verbose=False)

        f1 = f1.numpy()
        results[f"{emo}_bertscore_mean"] = f1.mean()
        results[f"{emo}_bertscore_max"] = f1.max()

    return results

In [ ]:
features = []

for text in tqdm(df[TEXT_COL], total=len(df)):
    row_feat = {}

    row_feat.update(tfidf_similarities(text))
    row_feat.update(embedding_similarities(text))
    row_feat.update(jaccard_similarities(text))

    features.append(row_feat)

features_df = pd.DataFrame(features)

df_final = pd.concat([df, features_df], axis=1)

In [ ]:
df_final.to_csv("../data/meta/go_emotions_meta.csv", index=False)